# Пример методом обработки результатов измерений. Измерения прямые многократные

In [1]:
# Пункт 5. Оценка измеряемой величины и среднее квадратическое отклонение
from moncenterlib.stats.stats_gost_R_8_736_2011.basic_stats import calc_basic_stats 

# Пункт 6. Исключение грубых погрешностей
from moncenterlib.stats.stats_gost_R_8_736_2011.grubbs_filter import grubbs_filter

# Пункт 7. Доверительные границы случайной погрешности
# при 15 < n <= 50
from moncenterlib.stats.stats_gost_R_8_736_2011.composite_test import check_normality_composite
# при n > 50
from moncenterlib.stats.stats_gost_R_8_736_2011.pearson_test import pearson_chi_square_normality
from moncenterlib.stats.stats_gost_R_8_736_2011.mises_smirnov_test import mises_smirnov_omega2_normality
# Доверительные границы случайной погрешности оценки измеряемой вели­чины
from moncenterlib.stats.stats_gost_R_8_736_2011.confidence_random_error import confidence_random_error

# Пункт 8. Доверительные границы неисключенной систематической погрешности
from moncenterlib.stats.stats_gost_R_8_736_2011.systematic_error import (
    systematic_error_confidence,
    SystematicComponent,
)

# Пункт 9. Доверительные границы погрешности оценки измеряемой величины
from moncenterlib.stats.stats_gost_R_8_736_2011.total_error import total_error_confidence

# Пункт 10. Форма записи оценки измеряемой величины
from moncenterlib.stats.stats_gost_R_8_736_2011.result_formatting import format_measurement_result

from pprint import pprint

## Пример 1. 15 < n <= 50. Нормальное распределение есть

In [2]:
values_16_50_ok = [
    10.081, 9.969, 9.974, 9.946, 10.043, 9.885, 10.087, 9.962,
    10.016, 9.988, 10.073, 9.897, 9.984, 9.981, 10.057, 9.945,
    9.991, 9.956, 10.002, 10.029, 9.945, 10.057, 10.045, 10.025,
    9.700, 10.340
]

result = calc_basic_stats(values_16_50_ok)
print(result)
print(result.S_x_mean)

BasicStats(n=26, x_mean=9.999153846153847, S=0.10503283003240177, S_x_mean=0.02059863268858042, S_biased=0.1029931634429021)
0.02059863268858042


In [3]:
result = grubbs_filter(values_16_50_ok, alpha=0.05) # 0.05 = 5%
cleaned_values = result.cleaned_values
print(cleaned_values)
print(result.removed)
# Просмотр хода выполнения фильтра Граббса
for check in result.checks:
    print(check)

[10.081, 9.969, 9.974, 9.946, 10.043, 9.885, 10.087, 9.962, 10.016, 9.988, 10.073, 9.897, 9.984, 9.981, 10.057, 9.945, 9.991, 9.956, 10.002, 10.029, 9.945, 10.057, 10.045, 10.025]
[10.34, 9.7]
GrubbsCheck(n=26, x_mean=9.999153846153847, S=0.10503283003240177, x_min=9.7, x_max=10.34, g_min=2.848193712971091, g_max=3.2451391982964224, g_crit=2.8407740764706424)
GrubbsCheck(n=25, x_mean=9.985520000000001, S=0.0803570780952121, x_min=9.7, x_max=10.087, g_min=3.5531406413471105, g_max=1.2628632399968405, g_crit=2.8216812378051954)
GrubbsCheck(n=24, x_mean=9.997416666666666, S=0.05518972546680633, x_min=9.885, x_max=10.087, g_min=2.0369129528335668, g_max=1.6231886021468462, g_crit=2.8015511615503152)


In [4]:
result = check_normality_composite(cleaned_values, q1_percent=5, q2_percent=5)

pprint(result)
pprint(result.passed)

CompositeNormalityResult(n=24,
                         x_mean=9.997416666666666,
                         S=0.05518972546680633,
                         S_biased=0.05402770636956151,
                         criterion_1=CompositeCriterion1Result(d=0.8381759150104341,
                                                               d_low=0.73376,
                                                               d_high=0.87188,
                                                               passed=True),
                         criterion_2=CompositeCriterion2Result(threshold=0.11976170426296974,
                                                               exceed_count=0,
                                                               allowed_exceed_count=2,
                                                               p_value_table=0.97,
                                                               z_value=2.17,
                                                               passed=True),

In [5]:
random_error = confidence_random_error(cleaned_values, p_conf=0.95)
pprint(random_error)

RandomErrorConfidenceResult(n=24,
                            p_conf=0.95,
                            df=23,
                            x_mean=9.997416666666666,
                            S=0.05518972546680633,
                            S_x_mean=0.011265555536496805,
                            t_value=2.0686576104190486,
                            delta=0.023304577196172564)


In [6]:
result = total_error_confidence(random_error)
pprint(result)

TotalErrorResult(p_conf=0.95,
                 x_mean=9.997416666666666,
                 delta_random=0.023304577196172564,
                 theta_systematic=0.0,
                 s_x_mean=0.011265555536496805,
                 s_theta=0.0,
                 s_total=0.011265555536496805,
                 k_total=2.0686576104190486,
                 delta_total=0.023304577196172564,
                 theta_mode='plain',
                 theta_k=None)


In [7]:
result = format_measurement_result(result.x_mean, result.delta_total, result.p_conf)
pprint(result)
pprint(result.notation)

RoundedMeasurementResult(x_raw=9.997416666666666,
                         delta_raw=0.023304577196172564,
                         p_conf=0.95,
                         x_rounded=9.997,
                         delta_rounded=0.023,
                         delta_significant_digits=2,
                         decimal_places=3,
                         notation='9.997 ± 0.023, P=0.95')
'9.997 ± 0.023, P=0.95'


## Пример 2. 15 < n <= 50. Нормальное распределение нету

In [8]:
values_16_50_fail = [
    9.90, 9.91, 9.92, 9.93, 9.94, 9.95, 9.96, 9.97,
    9.98, 9.99, 10.00, 10.01,
    10.20, 10.21, 10.22, 10.23, 10.24, 10.25,
    10.26, 10.27, 10.28, 10.29, 10.30, 10.31,
    9.30, 10.90
]

result = calc_basic_stats(values_16_50_fail)
print(result)
print(result.S_x_mean)

BasicStats(n=26, x_mean=10.104615384615386, S=0.27192985520873963, S_x_mean=0.053329832232224005, S_biased=0.26664916116112003)
0.053329832232224005


In [9]:
result = grubbs_filter(values_16_50_fail, alpha=0.05) # 0.05 = 5%
cleaned_values = result.cleaned_values
print(cleaned_values)
print(result.removed)
# Просмотр хода выполнения фильтра Граббса
for check in result.checks:
    print(check)

[9.9, 9.91, 9.92, 9.93, 9.94, 9.95, 9.96, 9.97, 9.98, 9.99, 10.0, 10.01, 10.2, 10.21, 10.22, 10.23, 10.24, 10.25, 10.26, 10.27, 10.28, 10.29, 10.3, 10.31]
[9.3, 10.9]
GrubbsCheck(n=26, x_mean=10.104615384615386, S=0.27192985520873963, x_min=9.3, x_max=10.9, g_min=2.9589078551075003, g_max=2.924962449503969, g_crit=2.8407740764706424)
GrubbsCheck(n=25, x_mean=10.136800000000001, S=0.22129768789272672, x_min=9.9, x_max=10.9, g_min=1.0700518485072856, g_max=3.4487481874187407, g_crit=2.8216812378051954)
GrubbsCheck(n=24, x_mean=10.105, S=0.15723148263129358, x_min=9.9, x_max=10.31, g_min=1.3038101312109562, g_max=1.3038101312109562, g_crit=2.8015511615503152)


In [10]:
result = check_normality_composite(cleaned_values, q1_percent=5, q2_percent=5)

pprint(result)
pprint(result.criterion_1.passed)
pprint(result.criterion_2.passed)
pprint(result.passed)

CompositeNormalityResult(n=24,
                         x_mean=10.105,
                         S=0.15723148263129358,
                         S_biased=0.15392097539538482,
                         criterion_1=CompositeCriterion1Result(d=0.9745260489331437,
                                                               d_low=0.73376,
                                                               d_high=0.87188,
                                                               passed=False),
                         criterion_2=CompositeCriterion2Result(threshold=0.34119231730990707,
                                                               exceed_count=0,
                                                               allowed_exceed_count=2,
                                                               p_value_table=0.97,
                                                               z_value=2.17,
                                                               passed=True),
         

## Пример 3. n > 50. Нормальное распределение есть

In [11]:
values_gt_50_ok = [
    9.975, 9.997, 9.872, 10.098, 9.892, 9.949, 10.030, 9.925,
    9.937, 9.945, 10.033, 10.138, 10.002, 9.933, 10.032, 9.964,
    9.999, 10.071, 9.955, 10.001, 9.947, 9.991, 10.015, 9.941,
    9.980, 9.986, 9.962, 9.929, 9.915, 9.991, 9.984, 10.134,
    9.854, 10.007, 10.022, 10.082, 10.030, 9.949, 10.000, 10.033,
    9.981, 10.046, 9.888, 10.104, 10.088, 9.980, 10.037, 10.003,
    9.950, 10.005, 10.060, 9.977, 9.977, 9.996, 10.026, 10.077,
    9.962, 10.031, 10.013, 9.888,
    9.550, 10.480
]

result = calc_basic_stats(values_gt_50_ok)
print(result)
print(result.S_x_mean)

BasicStats(n=62, x_mean=9.993854838709678, S=0.1036717465936445, S_x_mean=0.013166324983724418, S_biased=0.10283228543701534)
0.013166324983724418


In [12]:
result = grubbs_filter(values_gt_50_ok, alpha=0.05) # 0.05 = 5%
cleaned_values = result.cleaned_values
print(cleaned_values)
print(result.removed)
# Просмотр хода выполнения фильтра Граббса
for check in result.checks:
    print(check)

[9.975, 9.997, 9.872, 10.098, 9.892, 9.949, 10.03, 9.925, 9.937, 9.945, 10.033, 10.138, 10.002, 9.933, 10.032, 9.964, 9.999, 10.071, 9.955, 10.001, 9.947, 9.991, 10.015, 9.941, 9.98, 9.986, 9.962, 9.929, 9.915, 9.991, 9.984, 10.134, 9.854, 10.007, 10.022, 10.082, 10.03, 9.949, 10.0, 10.033, 9.981, 10.046, 9.888, 10.104, 10.088, 9.98, 10.037, 10.003, 9.95, 10.005, 10.06, 9.977, 9.977, 9.996, 10.026, 10.077, 9.962, 10.031, 10.013, 9.888]
[10.48, 9.55]
GrubbsCheck(n=62, x_mean=9.993854838709678, S=0.1036717465936445, x_min=9.55, x_max=10.48, g_min=4.281348132866199, g_max=4.689273377401792, g_crit=3.2121652713703654)
GrubbsCheck(n=61, x_mean=9.98588524590164, S=0.08320719086726733, x_min=9.55, x_max=10.138, g_min=5.238552598139822, g_max=1.8281443287878125, g_crit=3.2059772788453422)
GrubbsCheck(n=60, x_mean=9.993149999999998, S=0.06137460000214858, x_min=9.854, x_max=10.138, g_min=2.2672245520969208, g_max=2.3600968477991033, g_crit=3.199661829437385)


In [13]:
result = pearson_chi_square_normality(cleaned_values, alpha=0.05)
pprint(result)
pprint(result.passed)

PearsonNormalityResult(n=60,
                       r=8,
                       h=0.03550000000000009,
                       x_mean=9.993149999999998,
                       S=0.06137460000214858,
                       chi2_value=7.171171456849844,
                       df=5,
                       alpha=0.05,
                       chi2_low=0.8312116134866625,
                       chi2_high=12.832501994030027,
                       passed=True,
                       intervals=[PearsonInterval(index=1,
                                                  left=9.854,
                                                  right=9.8895,
                                                  center=9.871749999999999,
                                                  observed=4,
                                                  expected=1.9574979478944223,
                                                  y=-1.9780169646034285,
                                                  contribution=2.1311

In [14]:
result = mises_smirnov_omega2_normality(cleaned_values, alpha=0.1)
pprint(result)
pprint(result.passed)

MisesSmirnovNormalityResult(n=60,
                            x_mean=9.993149999999998,
                            S=0.06137460000214858,
                            n_omega2=0.2550644075616617,
                            a_value=0.027532203780830853,
                            alpha=0.1,
                            threshold=0.9,
                            passed=True,
                            rows=[MisesSmirnovRow(j=1,
                                                  x_j=9.854,
                                                  a_j=0.008333333333333333,
                                                  F_xj=0.011688255858051555,
                                                  ln_F_xj=-4.44917071411184,
                                                  one_minus_a_j=0.9916666666666667,
                                                  one_minus_F_xj=0.9883117441419484,
                                                  ln_one_minus_F_xj=-0.01175710049550613,
                  

In [15]:
random_error = confidence_random_error(cleaned_values, p_conf=0.95)
pprint(random_error)

RandomErrorConfidenceResult(n=60,
                            p_conf=0.95,
                            df=59,
                            x_mean=9.993149999999998,
                            S=0.06137460000214858,
                            S_x_mean=0.007923426789615438,
                            t_value=2.000995378088267,
                            delta=0.01585474038464125)


In [16]:
components = [
    SystematicComponent("калибровка", 0.05),
    SystematicComponent("температура", 0.02),
]
systematic_error = systematic_error_confidence(components, p_conf=0.95)
pprint(systematic_error)

SystematicErrorResult(m=2,
                      p_conf=0.95,
                      k=1.0,
                      theta_sum=0.07,
                      method='sum',
                      components=[SystematicComponent(name='калибровка',
                                                      theta=0.05,
                                                      influence_coefficient=1.0),
                                  SystematicComponent(name='температура',
                                                      theta=0.02,
                                                      influence_coefficient=1.0)])


In [17]:
total_error = total_error_confidence(random_error, systematic_error.theta_sum)
pprint(total_error)

TotalErrorResult(p_conf=0.95,
                 x_mean=9.993149999999998,
                 delta_random=0.01585474038464125,
                 theta_systematic=0.07,
                 s_x_mean=0.007923426789615438,
                 s_theta=0.04041451884327381,
                 s_total=0.041183904931705165,
                 k_total=1.7761354823947149,
                 delta_total=0.07314819485277223,
                 theta_mode='plain',
                 theta_k=None)


In [18]:
result = format_measurement_result(total_error.x_mean, total_error.delta_total, total_error.p_conf)
pprint(result)
pprint(result.notation)

RoundedMeasurementResult(x_raw=9.993149999999998,
                         delta_raw=0.07314819485277223,
                         p_conf=0.95,
                         x_rounded=9.99,
                         delta_rounded=0.07,
                         delta_significant_digits=1,
                         decimal_places=2,
                         notation='9.99 ± 0.07, P=0.95')
'9.99 ± 0.07, P=0.95'


# Расчет одной командой

In [19]:
from moncenterlib.stats.stats_gost_R_8_736_2011.calc_save import calculate_and_save_protocol
from moncenterlib.stats.stats_gost_R_8_736_2011.dataclasses import SystematicComponent

values_gt_50_ok = [
    9.975, 9.997, 9.872, 10.098, 9.892, 9.949, 10.030, 9.925,
    9.937, 9.945, 10.033, 10.138, 10.002, 9.933, 10.032, 9.964,
    9.999, 10.071, 9.955, 10.001, 9.947, 9.991, 10.015, 9.941,
    9.980, 9.986, 9.962, 9.929, 9.915, 9.991, 9.984, 10.134,
    9.854, 10.007, 10.022, 10.082, 10.030, 9.949, 10.000, 10.033,
    9.981, 10.046, 9.888, 10.104, 10.088, 9.980, 10.037, 10.003,
    9.950, 10.005, 10.060, 9.977, 9.977, 9.996, 10.026, 10.077,
    9.962, 10.031, 10.013, 9.888,
    9.550, 10.480
]
config = {
    "alpha_grubbs": 0.05,           # 0.01, 0.05
    "normal_n<15": False,           # Если распределение нормально для n < 15, то True, иначе False
    "normality_composite_q1": 5,    # 1 или 5 (1 и 99), (5, 95)
    "normality_composite_q2": 5,    # 1, 2, 5,
    "alpha_pearson": 0.05,          # [0.02; 0.10]
    "r_pearson": None,              # Автоматический расчет интервалов. Либо можно вписать свое количество интервало
    "alpha_smirnov": 0.1,           # 0.1 или 0.2,
    "random_error_p_conf": 0.95,    # 0.95, 0.99
    "systematic_components": [
        SystematicComponent("калибровка", 0.05),
        SystematicComponent("температура", 0.02),
    ],
    "systematic_error_p_conf": 0.95,
    "systematic_error_k": None,     # если m <=4 и P=0.99, то k подбирается по графику, Рисунок 1. m - количество составляющих НСП
}

result = calculate_and_save_protocol(values_gt_50_ok, config, "protocol.txt")